# Running inference at scale and in real-time


<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/computer-vision/deeplearning-cv-pcb-flow-5.png?raw=true" width="700px" style="float: right"/>

We have now deployed our model to our Registry. The registry provides governance and ACL, simplifying and accelerating all downstream pipeline developments.

Other teams don't have to worry about the model itself, they can focus on Ops tasks & model serving, while Data Scientists can release new models when they feel ready. 

Models are typically used in 2 ways:

- At scale, a cluster (in batch or streaming, including within Lakeflow Pipelines)
- For real-time, low-latencies use-cases, served behind a REST API.

Databricks provides and simplify both options.

<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=7474654018043568&notebook=%2F03-running-cv-inferences&demo_name=computer-vision-pcb&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fcomputer-vision-pcb%2F03-running-cv-inferences&version=1&user_hash=bbdbed5f785ffc3e7d74bf8bb0e97e3e6421d6ad205d21255f112e10dfba5ca0">

## Batch & Streaming Model scoring

Let's start with Batch / Streaming inference. We'll use spark distributed capabilities to run our inferences at scale.

To do so, we need to load our model from the MLFlow registry, and build a Pandas UDF to distribute the inference on multiple instances (typically on GPUs).

The first step consists of installing the model dependencies to make sure we're loading the model using the same library versions.

In [0]:
%pip install databricks-sdk mlflow==3.14.0 torch torchvision "huggingface-hub==0.36.2"
dbutils.library.restartPython()

In [0]:
%run ./_resources/00-init $reset_all_data=false

In [0]:
from mlflow.store.artifact.models_artifact_repo import ModelsArtifactRepository
import os

# Use the Unity Catalog model registry
mlflow.set_registry_uri("databricks-uc")
MODEL_NAME = f"{catalog}.{db}.dbdemos_pcb_classification"
MODEL_URI = f"models:/{MODEL_NAME}@Production"

# download model requirement from remote registry
requirements_path = ModelsArtifactRepository(MODEL_URI).download_artifacts(artifact_path="requirements.txt") 

if not os.path.exists(requirements_path):
  dbutils.fs.put("file:" + requirements_path, "", True)

In [0]:
%pip install -r $requirements_path

In [0]:
import torch
#Make sure to leverage the GPU when available
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

pipeline = mlflow.transformers.load_model(MODEL_URI, device=device.index)

### Running our model inference locally (non distributed)

Let's first run prediction locally on a standard pandas dataframe:

In [0]:
import io
from PIL import Image
from torchvision.transforms.functional import to_tensor, to_pil_image

#Call the pipeline and returns the main class with its probability
def predict_byte_series(content_as_byte_series, pipeline):
  #Transform as a list of PIL Images for our huggingface pipeline:
  image_list = content_as_byte_series.apply(lambda b: Image.open(io.BytesIO(b))).to_list()
  #the pipeline returns the probability for all the class
  predictions = pipeline.predict(image_list)
  #Filter & returns only the class with the highest score [{'score': 0.999038815498352, 'label': 'normal'}, ...]
  return pd.DataFrame([max(r, key=lambda x: x['score']) for r in predictions])  


df = spark.read.table("training_dataset_augmented").limit(50)
#Switch our model in inference mode
pipeline.model.eval()
with torch.set_grad_enabled(False):
  predictions = predict_byte_series(df.limit(10).toPandas()['content'], pipeline)
display(predictions)

### Distribute the inference with Spark and a Pandas UDF (batch/streaming inference)
 
Let's parallelize our inferences by wrapping the function using a pandas UDF:

In [0]:
import numpy as np
import torch
from typing import Iterator

#Only batch the inferences in our udf by 1000 as images can take some memory
try:
  spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", 1000)
except:
  pass

@pandas_udf("struct<score: float, label: string>")
def detect_damaged_pcb(images_iter: Iterator[pd.Series]) -> Iterator[pd.DataFrame]:
  #Switch pipeline to eval mode
  pipeline.model.eval()
  with torch.set_grad_enabled(False):
    for images in images_iter:
      yield predict_byte_series(images, pipeline)

In [0]:
display(df.limit(3).select('filename', 'content').withColumn("prediction", detect_damaged_pcb("content")))

## Realtime inferences with REST API

Many use-cases require real-time capabilities. Think about real-time analysis in our PCB manufacturing system. A picture is taken and we need to instantly check for potential defects. 

To be able to do that, we'll need to serve our inference behind a REST API. The system can then send our images, and the endpoint answer with the prediction.

To do that, we'll have to encode our image byte as base64 and send them over the REST API.

This implies that our model should decode the base64 back as PIL image. To make this easy, we can create a custom model wrapper having the transformers pipeline and a simple method transforming the base64 before calling our pipeline.

This is done as usual extending the `mlflow.pyfunc.PythonModel` class:

In [0]:
from io import BytesIO
import base64

# Model wrapper
class RealtimeCVModelWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, pipeline):
        self.pipeline = pipeline
        # instantiate model in evaluation mode
        self.pipeline.model.eval()

    #images will contain a series of images encoded in b64
    def predict(self, context, images):
        with torch.set_grad_enabled(False):
          #Convert the base64 to PIL images
          images = images['data'].apply(lambda b: Image.open(BytesIO(base64.b64decode(b)))).to_list()
          # Make the predictions
          predictions = self.pipeline(images)
          # Return the prediction with the highest score
          return pd.DataFrame([max(r, key=lambda x: x['score']) for r in predictions])

In [0]:
def to_base64(b):
  return base64.b64encode(b).decode("ascii")


# Build our final hugging face pipeline by loading the model from the registry
pipeline_cpu = mlflow.transformers.load_model(MODEL_URI, return_type="pipeline", device=torch.device("cpu"))

# Wrap our model as a PyFuncModel so that it can be used as a real-time serving endpoint
rt_model = RealtimeCVModelWrapper(pipeline_cpu)

# Let's try locally before deploying our endpoint to make sure it works as expected:
#     Transform our input as a pandas dataframe containing base64 as this is what our serverless model endpoint will receive.
pdf = df.toPandas()
df_input = pd.DataFrame(pdf["content"].apply(to_base64).to_list(), columns=["data"])

predictions = rt_model.predict(None, df_input)
display(predictions)

Now that our wrapper is ready, let's deploy it as a new model in the registry.

If real-time serving is your main usage, you would typically do that in the training step while registering your model. For this demo, we'll make 2 separate models: 1 for batch and 1 with the base64 wrapper for real-time inferences.

In [0]:
from mlflow.models.signature import infer_signature
DBDemos.init_experiment_for_batch("computer-vision-dl", "pcb")

with mlflow.start_run(run_name="hugging_face_rt") as run:
  signature = infer_signature(df_input, predictions)
  #log the model to MLFlow
  reqs = mlflow.pyfunc.log_model(
    name="model",
    python_model=rt_model,
    pip_requirements=requirements_path, 
    input_example=df_input, 
    signature = signature)
  mlflow.set_tag("dbdemos", "pcb_classification")
  mlflow.set_tag("rt", "true")

#Save the model in the registry
model_registered = mlflow.register_model(
  model_uri="runs:/"+run.info.run_id+"/model", 
  name="dbdemos_pcb_classification")

### Deploying a Model Endpoint Serving

Our new model wrapper is available in the registry. We can deploy this new version as a model endpoint to start our real-time model serving.


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput, EndpointCoreConfigInput, ServingModelWorkloadType, AiGatewayConfig, AiGatewayInferenceTableConfig

serving_endpoint_name = "dbdemos_pcb_classification_endpoint"

# Specify the model serving endpoint configuration
endpoint_config = EndpointCoreConfigInput(
    name=serving_endpoint_name,
    served_entities=[
        ServedEntityInput(
            entity_name=MODEL_NAME,
            entity_version=model_registered.version,
            workload_size="Small",
            workload_type=ServingModelWorkloadType.CPU,
            scale_to_zero_enabled=True
        )
    ]
)

#Set this to True to release a newer version (the demo won't update the endpoint to a newer model version by default)
force_update = False 

# Check existing endpoints to see if this one already exists
w = WorkspaceClient()
existing_endpoint = next(
    (e for e in w.serving_endpoints.list() if e.name == serving_endpoint_name), None
)
if existing_endpoint == None:
    print(f"Creating the endpoint {serving_endpoint_name}, this will take a few minutes to package and deploy the endpoint...")
    from datetime import timedelta
    w.serving_endpoints.create_and_wait(
      name=serving_endpoint_name, 
      config=endpoint_config, 
      ai_gateway=AiGatewayConfig(
        inference_table_config=AiGatewayInferenceTableConfig(
          catalog_name=catalog, schema_name=db, enabled=True)),
      timeout=timedelta(minutes=60))
else:
  print(f"Endpoint {serving_endpoint_name} already exists...")
  if force_update:
    print(f"Updating the version of {endpoint_config.served_entities[0].entity_name} to version {endpoint_config.served_entities[0].entity_version} on endpoint {serving_endpoint_name}...")
    from datetime import timedelta
    w.serving_endpoints.update_config_and_wait(
      served_entities=endpoint_config.served_entities, 
      name=serving_endpoint_name,
      timeout=timedelta(minutes=60))

### Our endpoint is ready

You can access and configure your endpoint using the [Model Serving UI](/#mlflow/endpoints/dbdemos_pcb_classification_endpoint) or the API. The Model Endpoint is serverless, stopping and starting almost instantly. In our case, we chose to scale it down to zero when not used (ideal for test/dev environments). 


*Note that Databricks Model Serving lets you host multiple model versions, simplifying A/B testing and new model deployment.*

In [0]:
import timeit
import mlflow
from mlflow import deployments

client = mlflow.deployments.get_deploy_client("databricks")

for i in range(3):
    input_slice = df_input[2*i:2*i+2]
    starting_time = timeit.default_timer()
    inferences = client.predict(
        endpoint=serving_endpoint_name, 
        inputs={
            "dataframe_records": input_slice.to_dict(orient='records')
        })
    print(f"Inference time, end-to-end: {round((timeit.default_timer() - starting_time)*1000)}ms")
    print("  "+str(inferences))

## Conclusion

We covered how Databricks makes it easy to deploy deep learning models at scale, including behind a REST endpoint with Databricks Serverless Model Serving.

### Next step: model explainability

As next step, let's discover how to explain and highlight the pixels our model considers as damaged.

Open the [04-explaining-inference notebook]($./04-explaining-inference) to discover how to use SHAP to analyze our prediction.

### Going further

Working with huggingface might not be enough for you. For deeper, custom integration, you can also leverage libraries like pytorch.

Open the [05-torch-lightning-training-and-inference]($./05-torch-lightning-training-and-inference) notebook to discover how to train and deploy a Pytorch model with [PyTorch Lightning](https://www.pytorchlightning.ai/index.html).